# Experiment: miniserve success timeline on nixpkgs unstable

Objective:
- Build a recent timeline for `miniserve.aarch64-darwin` on `nixpkgs/unstable`.
- For each recent eval, connect the eval to the resolved `nixpkgs` revision and classify the job as `success`, `failed`, `not-finished`, or `absent`.


In [4]:
# Setup: imports and small HTTP helpers
from __future__ import annotations

import json
import urllib.error
import urllib.request
from datetime import UTC, datetime
from pprint import pprint


def fetch_json(url: str, *, timeout: int = 120):
    request = urllib.request.Request(
        url,
        headers={
            "Accept": "application/json",
            "User-Agent": "hydra-miniserve-timeline",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.load(response)
    except urllib.error.HTTPError as error:
        body = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {error.code} for {url}\n{body}") from error
    except urllib.error.URLError as error:
        raise RuntimeError(f"failed to fetch {url}: {error.reason}") from error


def iso8601_utc(timestamp: int | None) -> str | None:
    if timestamp is None:
        return None
    return datetime.fromtimestamp(timestamp, UTC).isoformat().replace("+00:00", "Z")


def eval_entries(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        return payload.get("evals") or []
    return []


def build_entries(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        return payload.get("builds") or []
    return []


def build_label(build: dict | None) -> str:
    if build is None:
        return "absent"
    if build.get("finished") != 1:
        return "not-finished"
    if build.get("buildstatus") == 0:
        return "success"
    return "failed"


## Plan

- Use `nixpkgs/unstable` eval history as the time axis.
- For each eval, read `jobsetevalinputs["nixpkgs"]["revision"]` to recover the Git commit.
- Inspect `/eval/{id}/builds` and look for the exact job `miniserve.aarch64-darwin`.
- Record a compact timeline so it is obvious where the job succeeded, failed, or was absent.


In [5]:
# Minimal baseline: one job, one jobset, recent history only
BASE_URL = "https://hydra.nixos.org"
PROJECT = "nixpkgs"
JOBSET = "unstable"
PACKAGE = "miniserve"
SYSTEM = "aarch64-darwin"
JOB = f"{PACKAGE}.{SYSTEM}"
INPUT_NAME = "nixpkgs"
RECENT_EVAL_LIMIT = 20

EVALS_URL = f"{BASE_URL}/jobset/{PROJECT}/{JOBSET}/evals"
EVALS_URL, JOB


('https://hydra.nixos.org/jobset/nixpkgs/unstable/evals',
 'miniserve.aarch64-darwin')

In [6]:
# Fetch recent eval ids. Hydra may return either a list or a dict containing an `evals` list.
raw_evals = fetch_json(EVALS_URL)
recent_evals = eval_entries(raw_evals)[:RECENT_EVAL_LIMIT]

print(f"recent evals fetched: {len(recent_evals)}")
if recent_evals:
    print("sample eval entry keys:")
    pprint(sorted(recent_evals[0].keys()))

recent_evals[:3]


recent evals fetched: 20
sample eval entry keys:
['builds',
 'checkouttime',
 'evaltime',
 'flake',
 'hasnewbuilds',
 'id',
 'jobsetevalinputs',
 'timestamp']


[{'checkouttime': 30,
  'builds': [58543921,
   58543983,
   58544316,
   58544666,
   58544819,
   58544838,
   58544886,
   64535121,
   64609085,
   67816066,
   67816321,
   74954997,
   74960677,
   80455534,
   100197900,
   145347845,
   145348047,
   189915350,
   189919343,
   189944746,
   203881245,
   203886763,
   203909863,
   214645782,
   214645892,
   215064124,
   215064952,
   215065818,
   215066128,
   235009860,
   243112941,
   243115533,
   246729583,
   246729884,
   276683755,
   276684188,
   276686498,
   276694469,
   276697602,
   276700403,
   276705724,
   276710016,
   276724155,
   276739216,
   276745351,
   276749374,
   276751837,
   276758210,
   276760790,
   276761766,
   276762147,
   276773159,
   276779096,
   276788892,
   276790175,
   276791647,
   276804147,
   276810287,
   276827652,
   276832658,
   276837352,
   276846029,
   276850309,
   276852377,
   276858676,
   276858812,
   276867012,
   276903740,
   276904069,
   276910261,
  

In [8]:
[eval["jobsetevalinputs"]["nixpkgs"]["revision"] for eval in recent_evals]

['4505c68428b26ec83fba400c5d6bd21e4e318d44',
 '2b69405f19c7004b832a7410c8aefa9d859feea3',
 '6c9aee27f4ead3aebab33a244beb11e74bf4d9de',
 '9fbe055270246ccdafe157ed256c0807a37e3d2a',
 '90707b0b5769ef343228525fbd687ff76474503e',
 '9cf7092bdd603554bd8b63c216e8943cf9b12512',
 '7f0e59b726f43126c6e5449135abfe1ec91bbfb5',
 '8c61e176a116ba19728b8f6722ebb64905e7d2fe',
 '35740f0044d94033cdae5838524a7bb9ed342d65',
 'f2b33993c481335d262fc7d9cd54c6d2a639076d',
 '0c46193f275d8fdfe9af421ab541fead2d0ab276',
 'd8496ec1589b8f8fe4a32753b68a800656b45954',
 'f8573b9c935cfaa162dd62cc9e75ae2db86f85df',
 'a07d4ce6bee67d7c838a8a5796e75dff9caa21ef',
 '8098522cef20ff3b6470e1a273f94e2e9742d256',
 '41b5f40a8f35e45c8058acee1d90bf6bc3bfde2e',
 'e80236013dc8b77aa49ca90e7a12d86f5d8d64c9',
 'f82ce7af0b79ac154b12e27ed800aeb97413723c',
 'aa19b12c6262625f8eae222cabba6e314c0191f9',
 'c7cbcd91179a64a16d56d5c0b9b9b3c4ae0948be']

In [ ]:
# Build a timeline for the target job across recent unstable evals.
timeline = []

for eval_entry in recent_evals:
    eval_id = int(eval_entry["id"])
    eval_data = fetch_json(f"{BASE_URL}/eval/{eval_id}")
    builds_payload = fetch_json(f"{BASE_URL}/eval/{eval_id}/builds")
    builds = build_entries(builds_payload)

    matching_build = next((build for build in builds if build.get("job") == JOB), None)
    nixpkgs_input = (eval_data.get("jobsetevalinputs") or {}).get(INPUT_NAME) or {}
    revision = nixpkgs_input.get("revision")
    timestamp = eval_data.get("timestamp") or eval_entry.get("timestamp")

    timeline.append(
        {
            "eval_id": eval_id,
            "when": iso8601_utc(timestamp),
            "revision": revision,
            "status": build_label(matching_build),
            "build_id": matching_build.get("id") if matching_build else None,
            "buildstatus": matching_build.get("buildstatus") if matching_build else None,
        }
    )

timeline[:10]


## Results

- Key observations:
- Surprises or failure modes:
- Decision: continue, pivot, or stop:


In [ ]:
# Summarize recent outcomes and keep the most useful rows visible.
summary = {
    "checked_evals": len(timeline),
    "successes": sum(row["status"] == "success" for row in timeline),
    "failed": sum(row["status"] == "failed" for row in timeline),
    "not_finished": sum(row["status"] == "not-finished" for row in timeline),
    "absent": sum(row["status"] == "absent" for row in timeline),
}

summary, timeline[:10]


## Next steps

- Increase `RECENT_EVAL_LIMIT` once the initial pass looks correct.
- Collapse repeated `revision` values if you want a revision-level summary rather than an eval-level timeline.
- If the per-eval build lookup is too slow, switch to caching or a narrower time window before scaling up.
